In [ ]:
import paramiko
from datetime import datetime
from pathlib import Path
import socket
import sys

# SSH Configuration for Pi
PI_USER = "sdl1"
PI_PASS = "1144"
PI_HOST = "192.168.0.101"
PI_CAPTURE_DIR = "/tmp"
SSH_PORT = 22
CONNECT_TIMEOUT_S = 8
CONNECT_RETRIES = 3


def get_project_paths():
    proj_root = Path.cwd()
    while proj_root.name != "AC-OTFlex-monorepo" and proj_root.parent != proj_root:
        proj_root = proj_root.parent

    out_dir = proj_root / "data" / "out" / "images"
    out_dir.mkdir(parents=True, exist_ok=True)
    return proj_root, out_dir


def build_capture_paths(out_dir: Path):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"otflex-top_{timestamp}.jpg"
    remote_path = f"{PI_CAPTURE_DIR}/{filename}"
    local_path = out_dir / filename
    return timestamp, filename, remote_path, local_path


def check_ssh_port_reachable(host: str, port: int, timeout_s: int = 3):
    with socket.create_connection((host, port), timeout=timeout_s):
        pass


def connect_ssh_with_retry(client: paramiko.SSHClient):
    last_error = None
    for attempt in range(1, CONNECT_RETRIES + 1):
        try:
            print(f"  Attempt {attempt}/{CONNECT_RETRIES}")
            client.connect(
                PI_HOST,
                port=SSH_PORT,
                username=PI_USER,
                password=PI_PASS,
                timeout=CONNECT_TIMEOUT_S,
                banner_timeout=CONNECT_TIMEOUT_S,
                auth_timeout=CONNECT_TIMEOUT_S,
            )
            return
        except (socket.timeout, TimeoutError) as e:
            last_error = e
            if attempt == CONNECT_RETRIES:
                raise
            print("  Timeout; retrying...")

    raise last_error if last_error is not None else TimeoutError("Unknown timeout")


def build_capture_command(remote_path: str):
    return f"""python3 << 'EOF'
from picamera2 import Picamera2
import time

picam2 = Picamera2()
config = picam2.create_still_configuration(main={{\"size\": (2028, 1520)}})
picam2.configure(config)
picam2.start()
time.sleep(2)
picam2.capture_file(\"{remote_path}\")
picam2.close()
print(\"OK\")
EOF
"""


def execute_remote_capture(client: paramiko.SSHClient, remote_path: str):
    capture_cmd = build_capture_command(remote_path)
    _, stdout, stderr = client.exec_command(capture_cmd, timeout=30)
    exit_code = stdout.channel.recv_exit_status()
    error = stderr.read().decode()

    if exit_code != 0:
        raise RuntimeError(f"Image capture failed: {error}")


def download_and_cleanup(client: paramiko.SSHClient, remote_path: str, local_path: Path):
    sftp = client.open_sftp()
    sftp.get(remote_path, str(local_path))
    sftp.close()
    client.exec_command(f"rm {remote_path}")

In [ ]:
# Run capture workflow
proj_root, out_dir = get_project_paths()
print(f"Project root: {proj_root}")
print(f"Output directory: {out_dir}")

_, filename, remote_path, local_path = build_capture_paths(out_dir)

print(f"\n[0/4] Checking TCP reachability to {PI_HOST}:{SSH_PORT}...")
try:
    check_ssh_port_reachable(PI_HOST, SSH_PORT, timeout_s=3)
    print("✓ SSH port reachable")
except OSError as e:
    print(f"ERROR: Cannot reach {PI_HOST}:{SSH_PORT} ({type(e).__name__}: {e})")
    print("Hint: verify Pi power/network, IP address, and firewall rules.")
    raise

ssh = None
try:
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    print(f"\n[1/4] Testing SSH connection to {PI_HOST}...")
    connect_ssh_with_retry(ssh)
    print("✓ SSH connection successful")

    print(f"\n[2/4] Capturing image on {PI_HOST}...")
    execute_remote_capture(ssh, remote_path)
    print("✓ Image captured on Pi")

    print(f"\n[3/4] Downloading image from {PI_HOST}...")
    download_and_cleanup(ssh, remote_path, local_path)
    print(f"✓ Image saved to {local_path.name}")

    print(f"\n[4/4] Cleaning up remote file...")
    print("✓ Remote cleanup complete")
    print("\n✓ All done!")

except paramiko.ssh_exception.AuthenticationException:
    print("ERROR: Authentication failed. Check username/password.")
    raise
except paramiko.ssh_exception.NoValidConnectionsError as e:
    print(f"ERROR: SSH service not accepting connections on {PI_HOST}:{SSH_PORT} ({e})")
    raise
except (socket.timeout, TimeoutError):
    print(
        f"ERROR: Connection timed out after {CONNECT_RETRIES} attempts "
        f"(timeout={CONNECT_TIMEOUT_S}s each)."
    )
    print("Hint: confirm Pi IP, Wi-Fi/Ethernet connectivity, and that SSH is enabled.")
    raise
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")
    raise
finally:
    if ssh is not None:
        ssh.close()

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPyImage
from PIL import Image as PILImage

# Navigate to project root from current working directory
proj_root = Path.cwd()
while proj_root.name != "AC-OTFlex-monorepo" and proj_root.parent != proj_root:
    proj_root = proj_root.parent

out_dir = proj_root / "data" / "out" / "images"

# Find all otflex-top images
images = sorted(out_dir.glob("otflex-top_*.jpg"), key=lambda p: p.stat().st_mtime)

if not images:
    print(f"No images found in {out_dir}")
else:
    latest = images[-1]

    with PILImage.open(latest) as img:
        rotated = img.rotate(180, expand=True)
        rotated.save(latest)

    print(f"Showing rotated image: {latest.name}")
    display(IPyImage(filename=str(latest)))